# SDS 10 — Comprehensive Exam Synthesis & Algorithm Recognition Reference

A cross-topic notebook for fast recognition, parameter verification, sanity checks, and Spark-safe scaffolds. Local math/helpers use only Python standard library. PySpark cells are reference scaffolds and require Spark on the exam server.

## 0. Recognition map

Use the **requested output + guarantee** before choosing the algorithm.

- reproducible fixed fraction of entities → deterministic hash sample
- membership, no false negatives → Bloom filter
- exactly k uniform among all seen → reservoir
- unique count → HLL
- frequency of key → CMS
- second moment → AMS
- ones in last N → DGIM
- Jaccard ANN → MinHash LSH
- cosine ANN → random-hyperplane LSH
- global graph prestige → PageRank
- hubs/authorities → HITS
- bridge-removal communities → Girvan-Newman
- structural pair similarity → SimRank
- triangles/wedges → clustering coefficient

In [ ]:
import math, random
from collections import defaultdict, Counter, deque

## 1. Parameter calculators: verify, do not guess

In [ ]:
def bloom_parameters(n, target_fpr, round_m=None):
    m_exact = -n * math.log(target_fpr) / (math.log(2)**2)
    m = int(math.ceil(m_exact)) if round_m is None else int(round_m)
    k_real = (m/n) * math.log(2)
    k = max(1, round(k_real))
    actual = (1 - math.exp(-k*n/m))**k
    return {"m_exact":m_exact, "m":m, "k_real":k_real, "k":k, "actual_fpr":actual}

def cms_parameters(epsilon, delta):
    return math.ceil(math.e/epsilon), math.ceil(math.log(1/delta))

def hll_registers_for_rse(target_rse):
    need = (1.04/target_rse)**2
    p = math.ceil(math.log2(need))
    m = 2**p
    return {"m_required":need, "p":p, "m":m, "rse":1.04/math.sqrt(m)}

print(bloom_parameters(500,0.02,4096))
print('CMS eps=.01 delta=.01:', cms_parameters(.01,.01))
print('HLL target 2%:', hll_registers_for_rse(.02))

### Cosine random-hyperplane LSH solver

In [ ]:
def hyperplane_collision(cosine):
    return 1 - math.acos(cosine)/math.pi

def candidate_probability(cosine, r, b):
    p = hyperplane_collision(cosine)
    return 1 - (1 - p**r)**b

def find_lsh_params(s_hi, target_hi, s_lo, target_lo, r_max=100):
    p_hi = hyperplane_collision(s_hi)
    p_lo = hyperplane_collision(s_lo)
    feasible=[]
    for r in range(1,r_max+1):
        # b >= ln(1-target_hi)/ln(1-p_hi^r)
        # b <= ln(1-target_lo)/ln(1-p_lo^r)
        den_hi=math.log1p(-(p_hi**r))
        den_lo=math.log1p(-(p_lo**r))
        bmin=math.ceil(math.log(1-target_hi)/den_hi)
        bmax=math.floor(math.log(1-target_lo)/den_lo)
        if bmin <= bmax:
            feasible.append((r,bmin,bmax,r*bmin))
    return feasible

print('r=31,b=81700:', candidate_probability(.6,31,81700), candidate_probability(.4,31,81700))
print('first feasible:', find_lsh_params(.6,.80,.4,.05)[:3])

## 2. Reservoir proof helper / simulation sanity check

In [ ]:
def reservoir_sample(stream, k, seed=0):
    rng=random.Random(seed); R=[]
    for i,item in enumerate(stream, start=1):
        if i <= k:
            R.append(item)
        else:
            j=rng.randint(1,i)
            if j <= k:
                R[j-1]=item
    return R

def reservoir_empirical(n=20,k=4,trials=10000):
    counts=Counter()
    for t in range(trials):
        for x in reservoir_sample(range(n),k,seed=t): counts[x]+=1
    return [counts[i]/trials for i in range(n)]

vals=reservoir_empirical()
print('expected inclusion =',4/20)
print('min/max empirical =',min(vals),max(vals))

## 3. PageRank sanity: rank mass should stay near 1

In [ ]:
def pagerank_local(edges, d=.85, tol=1e-12, max_iter=500):
    nodes=sorted(set(x for e in edges for x in e)); N=len(nodes)
    out=defaultdict(list)
    for u,v in edges: out[u].append(v)
    r={v:1/N for v in nodes}
    for it in range(max_iter):
        incoming={v:0.0 for v in nodes}
        for u in nodes:
            if out[u]:
                share=r[u]/len(out[u])
                for v in out[u]: incoming[v]+=share
        D=sum(r[u] for u in nodes if not out[u])
        new={v:(1-d)/N+d*(incoming[v]+D/N) for v in nodes}
        residual=sum(abs(new[v]-r[v]) for v in nodes)
        r=new
        if residual<tol: break
    return r,residual,it+1

r,res,it=pagerank_local([('A','B'),('B','C'),('C','B'),('C','D')])
print('sum ranks=',sum(r.values()),'residual=',res,'iters=',it)

## 4. HITS sanity: normalize both vectors

In [ ]:
def hits_local(edges, tol=1e-12, max_iter=500):
    nodes=sorted(set(x for e in edges for x in e)); h={v:1.0 for v in nodes}; a={v:1.0 for v in nodes}
    for it in range(max_iter):
        na={v:0.0 for v in nodes}
        for u,v in edges: na[v]+=h[u]
        norm=math.sqrt(sum(x*x for x in na.values())) or 1.0
        na={v:x/norm for v,x in na.items()}
        nh={v:0.0 for v in nodes}
        for u,v in edges: nh[u]+=na[v]
        norm=math.sqrt(sum(x*x for x in nh.values())) or 1.0
        nh={v:x/norm for v,x in nh.items()}
        residual=sum(abs(na[v]-a[v]) for v in nodes)+sum(abs(nh[v]-h[v]) for v in nodes)
        a,h=na,nh
        if residual<tol: break
    return h,a,residual,it+1

h,a,res,it=hits_local([('H1','A1'),('H1','A2'),('H2','A2'),('H2','A3')])
print('hub norm',math.sqrt(sum(x*x for x in h.values())))
print('auth norm',math.sqrt(sum(x*x for x in a.values())))

## 5. Graph counting convention sanity

In [ ]:
def graph_stats(edges):
    adj=defaultdict(set)
    for u,v in edges:
        if u==v: continue
        adj[u].add(v); adj[v].add(u)
    W=sum(len(adj[v])*(len(adj[v])-1)//2 for v in adj)
    T=0
    nodes=sorted(adj)
    for i,u in enumerate(nodes):
        for v in [x for x in adj[u] if x>u]:
            T += sum(1 for w in adj[u].intersection(adj[v]) if w>v)
    C=0 if W==0 else 3*T/W
    return T,W,C

print('triangle:', graph_stats([('A','B'),('B','C'),('A','C')]))

## 6. Modularity bug detector: score partitions on ORIGINAL graph

In [ ]:
def modularity_summary(edges, communities):
    adj=defaultdict(set)
    for u,v in edges: adj[u].add(v); adj[v].add(u)
    m=len({tuple(sorted(e)) for e in edges})
    membership={v:i for i,c in enumerate(communities) for v in c}
    Q=0.0
    for cid,c in enumerate(communities):
        internal=sum(1 for u,v in {tuple(sorted(e)) for e in edges} if membership.get(u)==cid and membership.get(v)==cid)
        D=sum(len(adj[v]) for v in c)
        Q += internal/m - (D/(2*m))**2
    return Q

G0=[('A','B'),('B','C'),('A','C'),('D','E'),('E','F'),('D','F'),('C','D')]
partition=[['A','B','C'],['D','E','F']]
working=[e for e in G0 if set(e)!=set(('C','D'))]
print('correct Q on G0:',modularity_summary(G0,partition))
print('wrong Q on damaged working graph:',modularity_summary(working,partition))

## 7. Spark-safe core patterns (reference; requires PySpark)

In [ ]:
# from pyspark.sql import functions as F
#
# # Stable entity sample
# sampled = df.filter(F.pmod(F.xxhash64(F.concat_ws('::',F.col('id'),F.lit('seed42'))),F.lit(10000)) < 100)
#
# # Safe graph cleaning / canonical undirected edge
# edges = (raw.filter(F.col('type')=='link')
#             .select('src','dst')
#             .filter(F.col('src').isNotNull() & F.col('dst').isNotNull())
#             .filter(F.col('src') != F.col('dst'))
#             .select(F.least('src','dst').alias('u'),F.greatest('src','dst').alias('v'))
#             .distinct().cache())
#
# # Iterative rule: cache/checkpoint large reused state; collect only scalar residuals
# residual = new.join(old,'id').agg(F.sum(F.abs(F.col('new')-F.col('old')))).first()[0]


## 8. Exam answer scaffold

For almost any SDS answer, make sure you can point to all seven:

1. **Algorithm match** — why this method fits the requested output/guarantee.
2. **Core formula / recurrence**.
3. **Explicit parameters**.
4. **Numerical verification** after rounding.
5. **Scalability statement**.
6. **Spark-native implementation** for the expensive portion.
7. **Sanity check + interpretation**.

For critique questions: first state what is correct, then identify only verified limitations/errors.

## 9. Mixed recognition drills

Try these without looking at the answer:

- “same 0.2% of devices every run” → deterministic hash sampling
- “no false negatives, 1% false positives” → Bloom filter
- “exactly 10,000 uniform events from everything seen” → reservoir
- “distinct users” → HLL
- “frequency of key with additive epsilon N error” → CMS
- “second frequency moment” → AMS
- “ones in the last N bits” → DGIM
- “cosine nearest-neighbor candidates” → random-hyperplane LSH
- “global link prestige” → PageRank
- “good hubs and authorities” → HITS
- “remove high-betweenness bridges” → Girvan-Newman
- “similar if pointed to by similar nodes” → SimRank
- “fraction of wedges closed” → clustering coefficient
- “balanced cut using second eigenvector” → spectral partitioning
- “one node may be in several communities” → overlapping-community method

## 10. Two-minute pre-submit checklist

- requested output actually shown?
- parameters explicit?
- probability/error verified numerically?
- built-in Spark only?
- any dangerous `collect()` of large state?
- destination-only / zero-score nodes preserved?
- convergence or bounded-iteration approximation stated?
- probability mass / normalization sanity checked?
- original graph preserved for modularity?
- graph direction/weight convention stated?